# Variable Substitution

SLayer lets callers inject `{variable}` placeholders at query time. This is what
makes a model or query **reusable** across values (a region, a date window, a
threshold) without string-concatenating SQL in client code.

Substitution works across SLayer's **two expression layers**:

- **Mode B — the DSL** (query `filters`): you filter on dimension/measure names.
- **Mode A — raw SQL** (a stored model's `sql` and `filters`, and each `Column`'s
  `sql` and `filter`): the value binds as a literal of that SQL, *before*
  aggregation. This is the primitive for parameterizing hand-written model SQL
  (for example, importing a Cube `FILTER_PARAMS` cube).

This notebook demonstrates every surface end-to-end against the Jaffle Shop
demo. See also: [Variables in model SQL](../../concepts/models.md#variables-in-model-sql)
· [Queries](../../concepts/queries.md#variables) · [References](../../concepts/references.md).

In [1]:
import os
import sys

sys.path.insert(0, os.path.join(os.getcwd(), "..", "..", ".."))
sys.path.insert(0, os.path.join(os.getcwd(), "..", "jaffle_data"))

from setup_jaffle import ensure_jaffle_shop

from slayer.core.models import SlayerModel

engine, storage, models = ensure_jaffle_shop()


def save(model: dict) -> str:
    """Save ``model`` as a stored model; queries name it."""
    engine.run_sync(engine.save_model(SlayerModel.model_validate(model)))
    return model["name"]

## Mode B — query-level filters (the DSL side)

A query `filter` references dimension/measure names, and `{var}` placeholders are
filled from the query's `variables`. One template, many values — no string
building on the caller side.

In [2]:
# One filter template, reused with different values.
for region in ["Brooklyn", "Philadelphia"]:
    result = engine.execute_sync(query={
        "source_model": "orders",
        "measures": ["count(*)", "sum(order_total)"],
        "dimensions": ["stores.name"],
        "filters": ["stores.name = '{region}'"],
        "variables": {"region": region},
    })
    row = result.data[0]
    print(f"{region:<14}{row['orders._count']:>10,} orders   ${row['orders.order_total_sum']:>14,.2f}")

Brooklyn         256,381 orders   $  2,846,086.56
Philadelphia     192,749 orders   $  2,183,509.31


### Formula surfaces

Measure formulas, computed dimensions, `order` expressions and `date_range` bounds take
placeholders too; an unnamed measure is named from its template, so the result key
(`orders.order_total_sum_rate_100`) stays the same whatever the value.

In [3]:
# A fee at a given rate: one formula template, many rates.
for rate in [1, 2.5]:
    result = engine.execute_sync(query={
        "source_model": "orders",
        "measures": ["sum(order_total) * {rate} / 100"],
        "variables": {"rate": rate},
    })
    print(f"rate {rate:>4}%  ->  {result.data[0]}")

rate    1%  ->  {'orders.order_total_sum_rate_100': 74447.3927}
rate  2.5%  ->  {'orders.order_total_sum_rate_100': 186118.48175}


## Mode A — raw SQL surfaces

Everything below binds the value as a literal of the SQL *inside* a stored model,
so it takes effect **before** aggregation — something a post-aggregation query filter
can't do. There are four raw-SQL surfaces; we cover each in turn.

### `SlayerModel.filters` — a model-level `WHERE`

Raw SQL lives only in a stored model, so we save one first. The `{floor}` value
lands in the always-applied `WHERE`.

In [4]:
orders_over_floor = save({
    "name": "orders_over_floor",
    "sql_table": "orders",
    "data_source": "jaffle_shop",
    "filters": ["order_total >= {floor}"],  # Mode A: raw SQL, filled from {floor}
    "columns": [
        {"name": "id", "sql": "id", "type": "TEXT", "primary_key": True},
        {"name": "order_total", "sql": "order_total", "type": "DOUBLE"},
    ],
})

for floor in [0, 50]:
    result = engine.execute_sync(query={
        "source_model": orders_over_floor,
        "measures": ["count(*)"],
        "variables": {"floor": floor},
    })
    print(f"order_total >= {floor:>3}: {result.data[0]['orders_over_floor._count']:>10,} orders")

# The value was bound into the SQL before it ran:
where = [line.strip() for line in result.sql.splitlines() if ">=" in line][0]
print("\nGenerated WHERE:", where)

order_total >=   0:    662,407 orders
order_total >=  50:     14,977 orders

Generated WHERE: orders_over_floor.order_total >= 50


### `SlayerModel.sql` — a full raw-SQL model body

An `sql`-mode model's body can reference `{var}` anywhere. Below `{floor}` appears
twice — once in the `WHERE` and once as a projected scalar — both filled from the
same variable.

In [5]:
floored_orders = save({
    "name": "floored_orders",
    "data_source": "jaffle_shop",
    "sql": (
        "SELECT id, order_total, {floor} AS floor_used "
        "FROM orders WHERE order_total >= {floor}"
    ),
    "columns": [
        {"name": "id", "sql": "id", "type": "TEXT", "primary_key": True},
        {"name": "order_total", "sql": "order_total", "type": "DOUBLE"},
        {"name": "floor_used", "sql": "floor_used", "type": "DOUBLE"},
    ],
})

result = engine.execute_sync(query={
    "source_model": floored_orders,
    "measures": ["count(*)"],
    "dimensions": ["floor_used"],
    "variables": {"floor": 60},
})
print(result.data[0])

{'floored_orders.floor_used': 60, 'floored_orders._count': 9800}


### `Column.sql` — a derived column expression

A column's `sql` is raw SQL too. Here a stored model's derived column scales
`order_total` by a runtime `{mult}`.

In [6]:
orders_scaled = save({
    "name": "orders_scaled",
    "sql_table": "orders",
    "data_source": "jaffle_shop",
    "columns": [
        {"name": "id", "sql": "id", "type": "TEXT", "primary_key": True},
        {"name": "order_total", "sql": "order_total", "type": "DOUBLE"},
        {"name": "scaled_total", "sql": "order_total * {mult}", "type": "DOUBLE"},
    ],
})

result = engine.execute_sync(query={
    "source_model": orders_scaled,
    "measures": [
        {"formula": "sum(order_total)", "name": "raw"},
        {"formula": "sum(scaled_total)", "name": "scaled"},
    ],
    "variables": {"mult": 2},
})
row = result.data[0]
print(f"raw sum      = ${row['orders_scaled.raw']:,.2f}")
print(f"scaled (x2)  = ${row['orders_scaled.scaled']:,.2f}")

raw sum      = $7,444,739.27
scaled (x2)  = $14,889,478.54


### `Column.filter` — a CASE-WHEN at aggregation time

`Column.filter` decides which rows contribute to *that column's* aggregate — the
natural home for windowed sums (e.g. this-year vs last-year) with scalar bounds.
Only rows passing `order_total >= {floor}` feed `big_total`.

In [7]:
orders_big = save({
    "name": "orders_big",
    "sql_table": "orders",
    "data_source": "jaffle_shop",
    "columns": [
        {"name": "id", "sql": "id", "type": "TEXT", "primary_key": True},
        {"name": "order_total", "sql": "order_total", "type": "DOUBLE"},
        {"name": "big_total", "sql": "order_total", "filter": "order_total >= {floor}", "type": "DOUBLE"},
    ],
})

result = engine.execute_sync(query={
    "source_model": orders_big,
    "measures": [
        {"formula": "sum(order_total)", "name": "all_orders"},
        {"formula": "sum(big_total)", "name": "big_orders"},
    ],
    "variables": {"floor": 50},
})
row = result.data[0]
print(f"SUM over all orders            = ${row['orders_big.all_orders']:,.2f}")
print(f"SUM over orders >= $50 (CASE)  = ${row['orders_big.big_orders']:,.2f}")

SUM over all orders            = $7,444,739.27
SUM over orders >= $50 (CASE)  = $1,097,047.85


## Variable precedence

The same variable can be set in several places. Highest priority wins:

**runtime `variables=` kwarg > stage `variables` > outer query `variables` (when a query-backed model is the `source_model`) > model `query_variables`**

`query_variables` on a model are the lowest-priority *defaults*.

In [8]:
p_orders = save({
    "name": "p_orders",
    "data_source": "jaffle_shop",
    "query_variables": {"floor": 100},  # lowest-priority default
    "sql": "SELECT id, order_total FROM orders WHERE order_total >= {floor}",
    "columns": [
        {"name": "id", "sql": "id", "type": "TEXT", "primary_key": True},
        {"name": "order_total", "sql": "order_total", "type": "DOUBLE"},
    ],
})


def count(query_var=None, runtime=None):
    q = {"source_model": p_orders, "measures": ["count(*)"]}
    if query_var is not None:
        q["variables"] = {"floor": query_var}
    kwargs = {"variables": {"floor": runtime}} if runtime is not None else {}
    return engine.execute_sync(query=q, **kwargs).data[0]["p_orders._count"]


print(f"model default   (floor=100) : {count():>8,}")
print(f"query variable  (floor=50)  : {count(query_var=50):>8,}")
print(f"runtime kwarg   (floor=200) : {count(query_var=50, runtime=200):>8,}   <- overrides the query's 50")

model default   (floor=100) :      114
query variable  (floor=50)  :   14,977


runtime kwarg   (floor=200) :        0   <- overrides the query's 50


## Missing variables & escaping

A `{var}` with no value (once any variable is in play) **raises** — a
parameterized model is meant to fail loudly, not silently match nothing. And
every value binds as a literal: an embedded quote stays inside the literal you
wrote, and a value that would change the SQL's structure is rejected.

In [9]:
# Missing variable -> clear error.
try:
    engine.execute_sync(query={
        "source_model": orders_over_floor,
        "measures": ["count(*)"],
        "variables": {"unrelated": 1},  # 'floor' is missing
    })
except ValueError as e:
    print("Missing variable raises:")
    print(" ", e)

# A hostile value in a bare slot is rejected before any SQL runs.
try:
    engine.execute_sync(query={
        "source_model": orders_over_floor,
        "measures": ["count(*)"],
        "variables": {"floor": "0 OR 1=1"},
    })
except ValueError as e:
    print("\nA non-literal value raises:")
    print(" ", e)

# Embedded quote is escaped (O'Brien -> O''Brien) so the SQL stays valid.
result = engine.execute_sync(query={
    "source_model": "orders",
    "measures": ["count(*)"],
    "dimensions": ["stores.name"],
    "filters": ["stores.name = '{name}'"],
    "variables": {"name": "O'Brien"},
})
literal = [line.strip() for line in result.sql.splitlines() if "Brien" in line][0]
print("\nValue O'Brien is escaped in the generated SQL:")
print(" ", literal)

Missing variable raises:
  Undefined variable 'floor' in Model 'orders_over_floor' filters[0]: 'order_total >= {floor}'. Available variables: ['unrelated']

A non-literal value raises:
  variables supply literal values: variable 'floor' holds the text '0 OR 1=1' in a number or boolean slot; put the placeholder inside quotes ('{floor}') to use it as a string.



Value O'Brien is escaped in the generated SQL:
  stores.name = 'O''Brien'


## Optional blocks — a filter that vanishes when its value is absent

Every surface above **requires** its variables. A Cube `FILTER_PARAMS` pushdown,
though, is *optional*: it filters when the caller supplies a value and becomes a
no-op when they don't. SLayer expresses that with an **optional block**
`{? ... ?}` on a Mode-A surface — it renders (parenthesised) when every `{var}`
inside is supplied, and collapses to the neutral `(1=1)` otherwise. A **list**
value renders an injection-safe `IN`-list (write the parens; *string* elements
are auto-quoted, numbers stay unquoted). Open the `WHERE` with `1=1` so the
collapse leaves valid SQL. This is exactly how `slayer import-cube` represents an
optional Cube `FILTER_PARAMS` pushdown.

In [10]:
orders_by_store = save({
    "name": "orders_by_store",
    "data_source": "jaffle_shop",
    "sql": (
        "SELECT o.id, o.order_total, s.name AS store_name "
        "FROM orders o LEFT JOIN stores s ON o.store_id = s.id "
        "WHERE 1=1 AND {? s.name IN ({stores}) ?}"
    ),
    "columns": [
        {"name": "id", "sql": "id", "type": "TEXT", "primary_key": True},
        {"name": "order_total", "sql": "order_total", "type": "DOUBLE"},
        {"name": "store_name", "sql": "store_name", "type": "TEXT"},
    ],
})

# Omitted -> the block collapses to (1=1): every order counts.
allc = engine.execute_sync(query={"source_model": orders_by_store, "measures": ["count(*)"]})

# Supplied (a list) -> renders `s.name IN ('Brooklyn', 'Philadelphia')`.
some = engine.execute_sync(query={
    "source_model": orders_by_store,
    "measures": ["count(*)"],
    "variables": {"stores": ["Brooklyn", "Philadelphia"]},
})

print(f"no filter (block collapses to (1=1)) : {allc.data[0]['orders_by_store._count']:>10,}")
print(f"stores IN [Brooklyn, Philadelphia]   : {some.data[0]['orders_by_store._count']:>10,}")
clause = [ln.strip() for ln in some.sql.splitlines() if "IN (" in ln][0]
print("\nGenerated (list -> IN-list):", clause)

no filter (block collapses to (1=1)) :    662,407
stores IN [Brooklyn, Philadelphia]   :    449,130

Generated (list -> IN-list): s.name IN ('Brooklyn', 'Philadelphia')


## Scope & summary

| Surface | Layer | Example |
|---|---|---|
| query `filters` | Mode B (DSL) | `"stores.name = '{region}'"` |
| measures, computed dimensions, `order`, `date_range` | Mode B (DSL) | `"sum(order_total) * {rate} / 100"` |
| `SlayerModel.filters` | Mode A (raw SQL) | `"order_total >= {floor}"` |
| `SlayerModel.sql` | Mode A (raw SQL) | `"... WHERE order_total >= {floor}"` |
| `Column.sql` | Mode A (raw SQL) | `"order_total * {mult}"` |
| `Column.filter` | Mode A (raw SQL) | `"order_total >= {floor}"` |
| optional block | Mode A (raw SQL) | `"... AND {? s.name IN ({stores}) ?}"` |

**Contract:** raise-on-missing once any variable is in play — for *required*
placeholders; a `{var}` **inside a `{? ... ?}` block** instead collapses the
block to `(1=1)` when absent. A fully variable-free execution leaves braces as
literals (so raw brace literals like a Postgres array `'{1,2,3}'` survive); use
`{{` / `}}` for literal braces in a model that *does* use variables. A **list**
value renders an injection-safe `IN`-list body (*string* elements auto-quoted,
numbers unquoted) — together the primitives for representing Cube `FILTER_PARAMS`
pushdowns.

**Scope:** substitution applies to a query's **direct source model**. Nested
`source_queries` stages, join targets, and cross-model targets are a tracked
follow-up. Every value binds as a literal, spelled in the datasource dialect's own
escaping, so no value can change the structure of the SQL it binds into.

In [11]:
# Remove the demo models again, so the shared Jaffle Shop store keeps its own.
for name in [orders_over_floor, floored_orders, orders_scaled, orders_big, p_orders, orders_by_store]:
    engine.run_sync(storage.delete_model(name, data_source="jaffle_shop"))